In [4]:
%pip install -U sacrebleu sentencepiece sacremoses

  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.1 MB/s  0:00:003.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 3.0 MB/s  0:00:003.1 MB/s eta 0:00:01
Using cached click-8.5.1-py3-none-any.whl (126 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 3.5 MB/s  0:00:02m 3.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [sacrebleu]━ 6/8 [sacremoses]
Note: you may need to restart the kernel to use updated packages.


In [6]:
# ============================================================
# SenseForge — Notebook 19
# English-Hindi Neural Machine Translation
# Part 1: Data preparation and pretrained baseline
# ============================================================

import os
import re
import gc
import json
import time
import random
import hashlib
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import sacrebleu

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_ROOT = ROOT.parent

IITB = DATA_ROOT / "Data_IIT-B_Hugging_Face"
SAM = DATA_ROOT / "Dataset_Samanantar"

OUT = ROOT / "artifacts" / "translation_v1"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
SEED = 42

# Keep the experiment manageable on an 8 GB Mac.
IITB_TRAIN_LIMIT = 6000
SAM_TRAIN_LIMIT = 2000
VAL_LIMIT = 300
TEST_LIMIT = 300

MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Project:", ROOT)
print("Device:", DEVICE)
print("Translation model:", MODEL_NAME)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def normalize(text):
    return " ".join(
        unicodedata.normalize("NFKC", str(text)).split()
    ).strip()

def key(text):
    return normalize(text).casefold()

def valid_pair(en, hi):
    en = normalize(en)
    hi = normalize(hi)

    if not (8 <= len(en) <= 400):
        return False

    if not (5 <= len(hi) <= 400):
        return False

    # The English source should contain Latin characters.
    if not re.search(r"[A-Za-z]", en):
        return False

    # The Hindi target should contain Devanagari.
    if not re.search(r"[\u0900-\u097F]", hi):
        return False

    return True

def clean_pairs(pairs, blocked=None, limit=None):
    blocked = blocked or set()
    seen = set()
    output = []

    for en, hi in pairs:
        en, hi = normalize(en), normalize(hi)

        if not valid_pair(en, hi):
            continue

        source_key = key(en)

        if source_key in blocked or source_key in seen:
            continue

        seen.add(source_key)
        output.append({
            "en": en,
            "hi": hi
        })

        if limit and len(output) >= limit:
            break

    return output

def save_json(name, obj):
    path = OUT / name
    path.write_text(
        json.dumps(
            obj, indent=2, ensure_ascii=False,
            default=str
        ),
        encoding="utf-8"
    )
    return path

# ------------------------------------------------------------
# IIT-B schema inspection
# ------------------------------------------------------------

print("\n=== IIT-B SCHEMA ===")

iitb_files = {
    "train": IITB / "train-00000-of-00001.parquet",
    "validation": IITB / "validation-00000-of-00001.parquet",
    "test": IITB / "test-00000-of-00001.parquet"
}

for split, path in iitb_files.items():
    assert path.exists(), f"Missing: {path}"

    parquet = pq.ParquetFile(path)

    print(
        split,
        "rows:", parquet.metadata.num_rows,
        "schema:", parquet.schema_arrow
    )

    sample = next(
        parquet.iter_batches(batch_size=2)
    ).to_pylist()

    print("Sample:", sample[0])

# The Hugging Face IIT-B corpus usually stores a
# translation dictionary containing 'en' and 'hi'.
# We verify that assumption rather than silently
# inventing language-column mappings.

def iitb_pairs(path):
    parquet = pq.ParquetFile(path)

    for batch in parquet.iter_batches(
        batch_size=2048
    ):
        for row in batch.to_pylist():
            translation = row.get("translation")

            if not isinstance(translation, dict):
                raise ValueError(
                    "Unexpected IIT-B translation schema"
                )

            if not {"en", "hi"}.issubset(translation):
                raise ValueError(
                    "IIT-B translation dictionary lacks en/hi"
                )

            yield translation["en"], translation["hi"]

# ------------------------------------------------------------
# Prepare original validation/test first
# ------------------------------------------------------------
# The original IIT-B evaluation partitions remain the
# evaluation source. We do not use Samanantar for testing.

print("\n=== PREPARING EVALUATION DATA ===")

val_all = clean_pairs(
    iitb_pairs(iitb_files["validation"])
)

test_all = clean_pairs(
    iitb_pairs(iitb_files["test"])
)

# Ensure the evaluation source sentences do not overlap.
test_keys = {key(r["en"]) for r in test_all}

val_all = [
    r for r in val_all
    if key(r["en"]) not in test_keys
]

assert val_all and test_all

rng = random.Random(SEED)
rng.shuffle(val_all)
rng.shuffle(test_all)

val_rows = val_all[:VAL_LIMIT]
test_rows = test_all[:TEST_LIMIT]

# Block ALL original evaluation sources, not just
# the smaller evaluation samples.
blocked_sources = (
    {key(r["en"]) for r in val_all}
    | test_keys
)

print("Validation:", len(val_rows))
print("Test:", len(test_rows))
print("Protected evaluation sources:", len(blocked_sources))

# ------------------------------------------------------------
# IIT-B training subset
# ------------------------------------------------------------

print("\n=== IIT-B TRAINING SUBSET ===")

iitb_train = clean_pairs(
    iitb_pairs(iitb_files["train"]),
    blocked=blocked_sources,
    limit=IITB_TRAIN_LIMIT
)

print("IIT-B selected:", len(iitb_train))

# ------------------------------------------------------------
# Samanantar supplementary subset
# ------------------------------------------------------------

print("\n=== SAMANANTAR AUDIT ===")

sam_files = sorted(
    SAM.glob("train-*.parquet")
)

assert sam_files, "No Samanantar shards found"

print("Available shards:", len(sam_files))

for path in sam_files:
    print(
        path.name,
        "rows:", pq.ParquetFile(path).metadata.num_rows
    )

# Samanantar contains src/tgt. We must inspect
# actual language direction before using it.

sample_path = sam_files[0]
sample_batch = next(
    pq.ParquetFile(sample_path).iter_batches(
        batch_size=5
    )
).to_pylist()

print("\nSamanantar sample:")
for row in sample_batch[:3]:
    print({
        "src": str(row.get("src", ""))[:150],
        "tgt": str(row.get("tgt", ""))[:150]
    })

# Determine the direction using script evidence.
# Ambiguous direction must stop the experiment.

def devanagari_fraction(text):
    letters = [
        c for c in str(text)
        if c.isalpha()
    ]
    if not letters:
        return 0.0
    return sum(
        "\u0900" <= c <= "\u097F"
        for c in letters
    ) / len(letters)

src_dev = np.mean([
    devanagari_fraction(r["src"])
    for r in sample_batch
])

tgt_dev = np.mean([
    devanagari_fraction(r["tgt"])
    for r in sample_batch
])

print("Source Devanagari fraction:", src_dev)
print("Target Devanagari fraction:", tgt_dev)

if src_dev < 0.2 and tgt_dev > 0.5:
    SAM_DIRECTION = "src_en_tgt_hi"
elif tgt_dev < 0.2 and src_dev > 0.5:
    SAM_DIRECTION = "src_hi_tgt_en"
else:
    raise ValueError(
        "Samanantar language direction is ambiguous. "
        "Inspect samples before proceeding."
    )

print("Detected direction:", SAM_DIRECTION)

# Stream through the corpus rather than loading the
# entire multi-gigabyte dataset into memory.

sam_train = []
seen_sources = (
    blocked_sources
    | {key(r["en"]) for r in iitb_train}
)

for path in sam_files:
    parquet = pq.ParquetFile(path)

    for batch in parquet.iter_batches(
        batch_size=2048
    ):
        for row in batch.to_pylist():
            if SAM_DIRECTION == "src_en_tgt_hi":
                en, hi = row["src"], row["tgt"]
            else:
                en, hi = row["tgt"], row["src"]

            if not valid_pair(en, hi):
                continue

            source_key = key(en)

            if source_key in seen_sources:
                continue

            seen_sources.add(source_key)

            sam_train.append({
                "en": normalize(en),
                "hi": normalize(hi)
            })

            if len(sam_train) >= SAM_TRAIN_LIMIT:
                break

        if len(sam_train) >= SAM_TRAIN_LIMIT:
            break

    if len(sam_train) >= SAM_TRAIN_LIMIT:
        break

print("Samanantar selected:", len(sam_train))

train_rows = iitb_train + sam_train

rng.shuffle(train_rows)

assert train_rows and val_rows and test_rows

train_keys = {key(r["en"]) for r in train_rows}
val_keys = {key(r["en"]) for r in val_rows}
test_keys = {key(r["en"]) for r in test_rows}

assert not (train_keys & val_keys)
assert not (train_keys & test_keys)
assert not (val_keys & test_keys)

print("\n=== FINAL DATASET ===")
print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))
print("Exact source overlap: 0")

# ------------------------------------------------------------
# Save reproducible subsets
# ------------------------------------------------------------

for name, rows in [
    ("train", train_rows),
    ("validation", val_rows),
    ("test", test_rows)
]:
    path = OUT / f"{name}.jsonl"

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row, ensure_ascii=False
                ) + "\n"
            )

save_json("dataset_manifest.json", {
    "model": MODEL_NAME,
    "iitb_train": len(iitb_train),
    "samanantar_train": len(sam_train),
    "validation": len(val_rows),
    "test": len(test_rows),
    "samanantar_direction": SAM_DIRECTION,
    "seed": SEED,
    "source_overlap": 0,
    "limitations": [
        "Controlled subset experiment",
        "Samanantar direction inferred from script",
        "Near-duplicate and cross-corpus leakage not fully audited",
        "Source-language filtering is heuristic"
    ]
})

# ------------------------------------------------------------
# Load pretrained translation model
# ------------------------------------------------------------

print("\n=== PRETRAINED MODEL ===")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model.to(DEVICE)
model.eval()

def translate_batch(texts, batch_size=4):
    outputs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH
        ).to(DEVICE)

        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=2
            )

        outputs.extend(
            tokenizer.batch_decode(
                generated,
                skip_special_tokens=True
            )
        )

    return outputs

def translation_metrics(predictions, references):
    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references]
    ).score

    return {
        "sacrebleu": float(bleu),
        "chrf": float(chrf)
    }

# ------------------------------------------------------------
# Pretrained baseline on validation only
# ------------------------------------------------------------

print("\n=== PRETRAINED VALIDATION BASELINE ===")

baseline_val_pred = translate_batch(
    [r["en"] for r in val_rows]
)

baseline_val_metrics = translation_metrics(
    baseline_val_pred,
    [r["hi"] for r in val_rows]
)

print("Baseline validation:", baseline_val_metrics)

save_json(
    "pretrained_validation_metrics.json",
    baseline_val_metrics
)

pd.DataFrame({
    "source_en": [r["en"] for r in val_rows],
    "reference_hi": [r["hi"] for r in val_rows],
    "prediction_hi": baseline_val_pred
}).to_csv(
    OUT / "pretrained_validation_predictions.csv",
    index=False
)

print("\nCell 1 complete.")
print("Prepared data and baseline saved:", OUT)


Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Device: mps
Translation model: Helsinki-NLP/opus-mt-en-hi

=== IIT-B SCHEMA ===
train rows: 1659083 schema: translation: struct<en: string, hi: string>
  child 0, en: string
  child 1, hi: string
Sample: {'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}
validation rows: 520 schema: translation: struct<en: string, hi: string>
  child 0, en: string
  child 1, hi: string
Sample: {'translation': {'en': 'Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".', 'hi': "महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।"}}
test rows: 2507 schema: translation: struct<en: string, hi: string>
  child 0, en: string
  child 1, hi: string
Sample: {'translation': {'en': 'A bla

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


=== PRETRAINED VALIDATION BASELINE ===


model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

Baseline validation: {'sacrebleu': 7.099491459916793, 'chrf': 29.91392136783811}

Cell 1 complete.
Prepared data and baseline saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/translation_v1


In [8]:
# ============================================================
# SenseForge — Notebook 19
# Part 2: Neural MT fine-tuning, evaluation and export
# ============================================================

import copy
import gc
import json
import time
import math
import random

import torch
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
PATIENCE = 1

# Validation generation is expensive.
# Evaluate on a fixed validation subset each epoch.
EPOCH_VAL_LIMIT = min(100, len(val_rows))

TRAIN_OUT = OUT / "finetuned"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Tokenized translation dataset
# ------------------------------------------------------------

class TranslationDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.rows = rows
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]

        source = self.tokenizer(
            row["en"],
            truncation=True,
            max_length=MAX_SOURCE_LENGTH
        )

        target = self.tokenizer(
            text_target=row["hi"],
            truncation=True,
            max_length=MAX_TARGET_LENGTH
        )

        return {
            "input_ids": source["input_ids"],
            "attention_mask": source["attention_mask"],
            "labels": target["input_ids"]
        }

def collate_translation(batch):
    sources = [
        {
            "input_ids": r["input_ids"],
            "attention_mask": r["attention_mask"]
        }
        for r in batch
    ]

    encoded = tokenizer.pad(
        sources,
        padding=True,
        return_tensors="pt"
    )

    labels = [
        torch.tensor(
            r["labels"],
            dtype=torch.long
        )
        for r in batch
    ]

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    encoded["labels"] = labels
    return encoded

train_dataset = TranslationDataset(
    train_rows, tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation,
    num_workers=0
)

# ------------------------------------------------------------
# Optimizer and scheduler
# ------------------------------------------------------------

model.train()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

total_updates = updates_per_epoch * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(total_updates * 0.05)),
    num_training_steps=total_updates
)

# ------------------------------------------------------------
# Memory-safe smoke test
# ------------------------------------------------------------

print("=== TRAINING SMOKE TEST ===")

sample_batch = next(iter(train_loader))

sample_batch = {
    k: v.to(DEVICE)
    for k, v in sample_batch.items()
}

optimizer.zero_grad(set_to_none=True)

smoke_output = model(**sample_batch)
smoke_loss = smoke_output.loss

assert torch.isfinite(smoke_loss)

smoke_loss.backward()

print("Forward/backward successful.")
print("Smoke-test loss:", float(smoke_loss.item()))

optimizer.zero_grad(set_to_none=True)
del sample_batch, smoke_output, smoke_loss
gc.collect()

if DEVICE.type == "mps":
    torch.mps.empty_cache()

# ------------------------------------------------------------
# Fine-tuning
# ------------------------------------------------------------

print("\n=== STARTING ENGLISH-HINDI FINE-TUNING ===")
print("Training examples:", len(train_rows))
print("Epochs:", EPOCHS)
print("Device:", DEVICE)

history = []
best_chrf = -1.0
best_epoch = 0
no_improvement = 0

start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    steps = 0

    epoch_start = time.perf_counter()

    for step, batch in enumerate(train_loader, start=1):
        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        output = model(**batch)
        loss = output.loss

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")

        total_loss += float(loss.detach().item())

        (loss / GRAD_ACCUM).backward()

        if (
            step % GRAD_ACCUM == 0
            or step == len(train_loader)
        ):
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        steps += 1

        if step % 250 == 0:
            print(
                f"Epoch {epoch} | "
                f"{step}/{len(train_loader)} batches"
            )

        del batch, output, loss

    # Evaluate with deterministic generation.
    model.eval()

    eval_rows = val_rows[:EPOCH_VAL_LIMIT]

    val_predictions = translate_batch(
        [r["en"] for r in eval_rows]
    )

    val_metrics = translation_metrics(
        val_predictions,
        [r["hi"] for r in eval_rows]
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": total_loss / max(steps, 1),
        "validation_bleu": val_metrics["sacrebleu"],
        "validation_chrf": val_metrics["chrf"],
        "epoch_seconds": time.perf_counter() - epoch_start
    }

    history.append(epoch_result)

    print(
        f"\nEpoch {epoch:02d} | "
        f"Loss={epoch_result['train_loss']:.4f} | "
        f"BLEU={epoch_result['validation_bleu']:.2f} | "
        f"chrF={epoch_result['validation_chrf']:.2f}"
    )

    # Only validation performance controls checkpoint selection.
    if val_metrics["chrf"] > best_chrf:
        best_chrf = val_metrics["chrf"]
        best_epoch = epoch
        no_improvement = 0

        model.save_pretrained(
            TRAIN_OUT / "best_model",
            safe_serialization=True
        )

        tokenizer.save_pretrained(
            TRAIN_OUT / "best_model"
        )

        print("New best checkpoint saved.")
    else:
        no_improvement += 1

    if no_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

training_seconds = time.perf_counter() - start_time

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print("Best validation chrF:", best_chrf)
print("Training seconds:", round(training_seconds, 2))

save_json("training_history.json", history)

# ------------------------------------------------------------
# Reload selected checkpoint
# ------------------------------------------------------------

del model
gc.collect()

if DEVICE.type == "mps":
    torch.mps.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(
    TRAIN_OUT / "best_model"
).to(DEVICE)

model.eval()

print("\nBest checkpoint reloaded.")

# ------------------------------------------------------------
# Full validation evaluation
# ------------------------------------------------------------

print("\n=== FINAL VALIDATION EVALUATION ===")

final_val_pred = translate_batch(
    [r["en"] for r in val_rows]
)

final_val_metrics = translation_metrics(
    final_val_pred,
    [r["hi"] for r in val_rows]
)

print(final_val_metrics)

# ------------------------------------------------------------
# Final test evaluation
# ------------------------------------------------------------
# Test is used only after model selection is complete.

print("\n=== FINAL HELD-OUT TEST ===")

test_predictions = translate_batch(
    [r["en"] for r in test_rows]
)

test_metrics = translation_metrics(
    test_predictions,
    [r["hi"] for r in test_rows]
)

print(test_metrics)

pd.DataFrame({
    "source_en": [r["en"] for r in test_rows],
    "reference_hi": [r["hi"] for r in test_rows],
    "prediction_hi": test_predictions
}).to_csv(
    TRAIN_OUT / "test_predictions.csv",
    index=False
)

# ------------------------------------------------------------
# Save final research results
# ------------------------------------------------------------

results = {
    "model": MODEL_NAME,
    "training_examples": len(train_rows),
    "validation_examples": len(val_rows),
    "test_examples": len(test_rows),
    "best_epoch": best_epoch,
    "training_seconds": training_seconds,
    "pretrained_validation": baseline_val_metrics,
    "finetuned_validation": final_val_metrics,
    "finetuned_test": test_metrics,
    "dataset_sources": {
        "IIT-B": len(iitb_train),
        "Samanantar": len(sam_train)
    },
    "limitations": [
        "Controlled subset experiment",
        "Samanantar source direction inferred heuristically",
        "Test evaluated after validation-based checkpoint selection",
        "Near-duplicate/source-level leakage not fully audited"
    ]
}

save_json("final_translation_results.json", results)

# ------------------------------------------------------------
# Translation examples
# ------------------------------------------------------------

print("\n=== TRANSLATION EXAMPLES ===")

for i in range(min(5, len(test_rows))):
    print("\nEnglish:", test_rows[i]["en"])
    print("Reference:", test_rows[i]["hi"])
    print("Predicted:", test_predictions[i])

print("\n" + "=" * 65)
print("SENSEFORGE TRANSLATION EXPERIMENT COMPLETE")
print("=" * 65)

print("Checkpoint:", TRAIN_OUT / "best_model")
print("Final results:", OUT / "final_translation_results.json")
print("Test chrF:", test_metrics["chrf"])
print("Test SacreBLEU:", test_metrics["sacrebleu"])

=== TRAINING SMOKE TEST ===
Forward/backward successful.
Smoke-test loss: 0.8947319984436035

=== STARTING ENGLISH-HINDI FINE-TUNING ===
Training examples: 8000
Epochs: 2
Device: mps
Epoch 1 | 250/8000 batches
Epoch 1 | 500/8000 batches
Epoch 1 | 750/8000 batches
Epoch 1 | 1000/8000 batches
Epoch 1 | 1250/8000 batches
Epoch 1 | 1500/8000 batches
Epoch 1 | 1750/8000 batches
Epoch 1 | 2000/8000 batches
Epoch 1 | 2250/8000 batches
Epoch 1 | 2500/8000 batches
Epoch 1 | 2750/8000 batches
Epoch 1 | 3000/8000 batches
Epoch 1 | 3250/8000 batches
Epoch 1 | 3500/8000 batches
Epoch 1 | 3750/8000 batches
Epoch 1 | 4000/8000 batches
Epoch 1 | 4250/8000 batches
Epoch 1 | 4500/8000 batches
Epoch 1 | 4750/8000 batches
Epoch 1 | 5000/8000 batches
Epoch 1 | 5250/8000 batches
Epoch 1 | 5500/8000 batches
Epoch 1 | 5750/8000 batches
Epoch 1 | 6000/8000 batches
Epoch 1 | 6250/8000 batches
Epoch 1 | 6500/8000 batches
Epoch 1 | 6750/8000 batches
Epoch 1 | 7000/8000 batches
Epoch 1 | 7250/8000 batches
Epoch 1 

/Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61949]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


New best checkpoint saved.
Epoch 2 | 250/8000 batches
Epoch 2 | 500/8000 batches
Epoch 2 | 750/8000 batches
Epoch 2 | 1000/8000 batches
Epoch 2 | 1250/8000 batches
Epoch 2 | 1500/8000 batches
Epoch 2 | 1750/8000 batches
Epoch 2 | 2000/8000 batches
Epoch 2 | 2250/8000 batches
Epoch 2 | 2500/8000 batches
Epoch 2 | 2750/8000 batches
Epoch 2 | 3000/8000 batches
Epoch 2 | 3250/8000 batches
Epoch 2 | 3500/8000 batches
Epoch 2 | 3750/8000 batches
Epoch 2 | 4000/8000 batches
Epoch 2 | 4250/8000 batches
Epoch 2 | 4500/8000 batches
Epoch 2 | 4750/8000 batches
Epoch 2 | 5000/8000 batches
Epoch 2 | 5250/8000 batches
Epoch 2 | 5500/8000 batches
Epoch 2 | 5750/8000 batches
Epoch 2 | 6000/8000 batches
Epoch 2 | 6250/8000 batches
Epoch 2 | 6500/8000 batches
Epoch 2 | 6750/8000 batches
Epoch 2 | 7000/8000 batches
Epoch 2 | 7250/8000 batches
Epoch 2 | 7500/8000 batches
Epoch 2 | 7750/8000 batches
Epoch 2 | 8000/8000 batches

Epoch 02 | Loss=1.2056 | BLEU=7.68 | chrF=29.33
Early stopping triggered.

Trai

In [10]:
# Compare pretrained and fine-tuned models on the same test set.
# This is diagnostic evaluation, not model selection.

import gc
import json
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM

# Fine-tuned predictions already exist from Cell 2.
finetuned_predictions = list(test_predictions)

finetuned_metrics = translation_metrics(
    finetuned_predictions,
    [r["hi"] for r in test_rows]
)

# Release the fine-tuned model before loading the baseline.
model.to("cpu")
del model
gc.collect()

if DEVICE.type == "mps":
    torch.mps.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(DEVICE)

model.eval()

baseline_test_predictions = translate_batch(
    [r["en"] for r in test_rows]
)

baseline_test_metrics = translation_metrics(
    baseline_test_predictions,
    [r["hi"] for r in test_rows]
)

comparison = {
    "pretrained": baseline_test_metrics,
    "finetuned": finetuned_metrics
}

print("=== SAME-SET TEST COMPARISON ===")

for name, metrics in comparison.items():
    print(name, metrics)

print("\nDifference (fine-tuned - pretrained):")

for metric in ["sacrebleu", "chrf"]:
    difference = (
        finetuned_metrics[metric]
        - baseline_test_metrics[metric]
    )
    print(f"{metric}: {difference:+.4f}")

comparison_df = pd.DataFrame({
    "english": [r["en"] for r in test_rows],
    "reference_hindi": [r["hi"] for r in test_rows],
    "pretrained": baseline_test_predictions,
    "finetuned": finetuned_predictions
})

comparison_df.to_csv(
    OUT / "pretrained_vs_finetuned_test.csv",
    index=False
)

(OUT / "pretrained_vs_finetuned_metrics.json").write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8"
)

display(comparison_df.head(10))

print("\nComparison saved.")

=== SAME-SET TEST COMPARISON ===
pretrained {'sacrebleu': 9.659755539671227, 'chrf': 31.825381284077636}
finetuned {'sacrebleu': 9.922889125218603, 'chrf': 31.93097172277915}

Difference (fine-tuned - pretrained):
sacrebleu: +0.2631
chrf: +0.1056


,english,reference_hindi,pretrained,finetuned
0,Earlier research had revealed that happy dogs ...,पिछले शोधों से पता चला है कि खुश कुत्ते अपनी द...,पहले शोध ने प्रकट किया था कि खुश कुत्ते अपनी प...,पहले शोध ने प्रकट किया था कि खुश कुत्ते अपनी प...
1,Presenting CAT's trendy footwear and clothing ...,कैट की ट्रैडी फुटवियर और परिधानों की सिलेक्शन पेश,सी.टी. के चलन और कपड़े संग्रह प्रस्तुत करते हैं.,Cas Tecowass के चलन और पोशाक संग्रह को प्रस्तु...
2,Nita is celebrating her birthday in Jodhpur an...,नीता अपना बर्थ-डे मुंबई में नहीं बल्कि जोधपुर ...,नताता अपने जन्मदिन है योधध में और नहीं मुंबई में।,नताता ने अपने जन्मदिन को योगे में मना कर दिया ...
3,"Miss Hall, from Halifax, West Yorkshire, said:...","हैलीफैक्स, पश्चिमी यॉर्कशायर से मिस हॉल ने कहा...","मिस हॉल, Wagalth, पश्चिम Wagigt ने कहा: ""देख ब...","मिस हॉल, पश्चिम विंग से कहा, ""एंड को देखकर बचा..."
4,I will follow the facts wherever they lead.,"मैं तथ्यों का अनुपालन करूंगा, वे हमें जहां पर ...",मैं तथ्यों का पालन करेंगे जहाँ वे नेतृत्व.,"जहां कहीं वे नेतृत्व करेंगे, मैं तथ्यों का पाल..."
5,"In addition to these, Karnataka, Tamilnadu and...","इनके अलावा कर्नाटक, तमिलनाडु और पूर्वोत्तर के ...","इन के अलावा, कर्नानाका, तमिल और उत्तर-पूर्व मे...","इन के अलावा, कर्नानाका, तमिल और उत्तर-पूर्व मे..."
6,It is the duty of everyone to earn money for t...,परिवार के भरण-पोषण के लिए धन कमाना हर व्यक्ति ...,यह परिवार की भलाई के लिए पैसा कमाने का कर्तव्य...,परिवार की भलाई के लिए पैसा कमाने के लिए हर किस...
7,"In the meantime, a lot of people gathered there.",इसी बीच वहां और कई लोग आ गए।,"इस बीच, बहुत सारे लोग वहाँ इकट्ठा हुए ।","इस बीच, बहुत सारे लोग वहां इकट्ठा हुए।"
8,According to sources in the Petroleum Ministry...,पेट्रोलियम मंत्रालय के सूत्रों के मुताबिक दो आ...,यह फैसला है कि PDS की दुकानों से बिक्री को रोक...,PDS की दुकानों के माध्यम से बिक्री को रोकने के...
9,He said it was intended to avoid contamination...,उन्होंने कहा कि इसका उद्देश्य साक्ष्य के सम्मि...,उन्होंने कहा कि यह सबूतों के फैलाव से बचने का ...,उन्होंने कहा कि यह सबूतों से दूषित होने से बचन...



Comparison saved.
